<a href="https://colab.research.google.com/github/Sarahnjunge/snake-game/blob/main/math_visualizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-Powered Math Visualizer
**Steps 1–2: dataset + exploration**

Run the cells top to bottom (Runtime → Run all). The first three cells write the project modules into Colab's working folder:
- `mathlabel.py`: normalizer + sympy rule-based classifier (the *referee*)
- `generate_dataset.py`: synthetic labeled data
- `test_mathlabel.py`: edge-case tests

In [ ]:
%%writefile mathlabel.py
"""
mathlabel.py -- normalization + rule-based function-type classifier.

  normalize(text)  -> clean python-style string ("y = 2x² - 4x" -> "2x**2 - 4x")
  parse(text)      -> sympy expression in x (safe: whitelisted tokens only)
  classify(text)   -> (label, info)

Labels: linear, quadratic, exponential, logarithmic, trigonometric, other
        (+ "invalid" when the text can't be parsed safely)

This is the *referee*: it labels external data (e.g. Hugging Face) and checks
generated labels. It will also be the math engine behind the visualizer.
The ML model should NOT be trained on features derived from these same rules,
otherwise it just learns to imitate them.
"""
import re
import numpy as np
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations,
    implicit_multiplication_application, convert_xor,
)

x = sp.Symbol("x", real=True)

MAX_LEN = 200
ALLOWED_NAMES = {
    "x", "e", "pi", "sin", "cos", "tan", "cot", "sec", "csc",
    "exp", "log", "ln", "log10", "log2", "sqrt", "abs",
}
TRIG = (sp.sin, sp.cos, sp.tan, sp.cot, sp.sec, sp.csc)

_SUPERSCRIPTS = {"⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",
                 "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9", "⁻": "-"}
_TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)


class ParseError(ValueError):
    pass


def _replace_frac(s):
    """\\frac{a}{b} -> ((a)/(b)); handles one level of nesting via repetition."""
    pat = re.compile(r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}")
    while True:
        new = pat.sub(r"((\1)/(\2))", s)
        if new == s:
            return s
        s = new


def normalize(text: str) -> str:
    """Make assorted notations look like plain python-ish math. Does not parse."""
    s = text.strip()
    # unicode
    s = s.replace("−", "-").replace("–", "-").replace("×", "*").replace("·", "*")
    s = s.replace("π", "pi").replace("√", "sqrt")
    s = re.sub(r"([⁰¹²³⁴⁵⁶⁷⁸⁹⁻]+)",
               lambda m: "**" + "".join(_SUPERSCRIPTS[c] for c in m.group(1)), s)
    # latex
    s = s.replace("$", "")
    s = _replace_frac(s)
    s = re.sub(r"\\(?:left|right)", "", s)
    s = s.replace("\\cdot", "*").replace("\\times", "*")
    s = re.sub(r"\\(sin|cos|tan|cot|sec|csc|ln|log|exp|sqrt|pi)", r"\1", s)
    s = re.sub(r"\^\s*\{([^{}]*)\}", r"^(\1)", s)           # x^{2} -> x^(2)
    s = re.sub(r"\\log_\{?(\d+)\}?\s*", lambda m: f"log{m.group(1)}", s)  # \log_2 -> log2
    s = s.replace("{", "(").replace("}", ")")
    # strip "y =", "f(x) ="
    s = re.sub(r"^\s*(?:y|f\s*\(\s*x\s*\)|g\s*\(\s*x\s*\))\s*=\s*", "", s, flags=re.I)
    # sin^2(x) style is ambiguous -> leave; users rarely type it in this tool
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _check_safe(s: str):
    if not s or len(s) > MAX_LEN:
        raise ParseError("empty or too long")
    if re.search(r"[^0-9A-Za-z_+\-*/^().,\s]", s):
        raise ParseError("illegal character")
    if "__" in s:
        raise ParseError("illegal token")
    for name in re.findall(r"[A-Za-z_]+\d*", s):
        if name not in ALLOWED_NAMES:
            raise ParseError(f"name not allowed: {name}")


def parse(text: str):
    """Return (normalized_string, sympy_expr). Raises ParseError."""
    s = normalize(text)
    _check_safe(s)
    local = {
        "x": x, "e": sp.E, "pi": sp.pi, "ln": sp.log, "abs": sp.Abs,
        "log10": lambda a: sp.log(a, 10), "log2": lambda a: sp.log(a, 2),
    }
    try:
        expr = parse_expr(s, local_dict=local, transformations=_TRANSFORMS, evaluate=True)
    except Exception as ex:  # SyntaxError, TypeError, ...
        raise ParseError(f"parse failed: {ex}") from ex
    expr = sp.sympify(expr)
    if not expr.is_number and not expr.free_symbols <= {x}:
        raise ParseError("extra variables")
    return s, expr


# ---------------------------------------------------------------- classification
def _linear_in_x(arg) -> bool:
    try:
        return arg.has(x) and sp.Poly(arg, x).degree() == 1
    except sp.PolynomialError:
        return False


def _is_nonconstant(expr) -> bool:
    pts = np.linspace(0.3, 6.0, 40)
    f = sp.lambdify(x, expr, "numpy")
    with np.errstate(all="ignore"):
        try:
            vals = np.array(f(pts), dtype=complex)
        except Exception:
            return True  # can't evaluate numerically; trust the symbolic result
    vals = vals[np.isfinite(vals)]
    if vals.size < 2:
        return True
    return bool(np.ptp(vals.real) > 1e-9 or np.ptp(vals.imag) > 1e-9)


def classify_expr(expr):
    """Classify a sympy expression. Returns (label, reason)."""
    if not expr.has(x):
        return "other", "constant"

    # --- polynomials ---------------------------------------------------------
    if expr.is_polynomial(x):
        deg = sp.Poly(sp.expand(expr), x).degree()
        if deg == 1:
            return "linear", "polynomial degree 1"
        if deg == 2:
            return "quadratic", "polynomial degree 2"
        return "other", f"polynomial degree {deg}"

    # --- find outermost transcendental atoms ---------------------------------
    fams, subs = set(), {}
    for node in sp.preorder_traversal(expr):
        if node in subs or not node.has(x):
            continue
        fam, arg = None, None
        if isinstance(node, TRIG):
            fam, arg = "trigonometric", node.args[0]
        elif isinstance(node, sp.exp):
            fam, arg = "exponential", node.args[0]
        elif isinstance(node, sp.Pow) and node.exp.has(x) and not node.base.has(x):
            base = node.base
            if not (base.is_positive and base != 1):
                return "other", "exponential base not positive / is 1"
            fam, arg = "exponential", node.exp
        elif isinstance(node, sp.log):
            fam, arg = "logarithmic", node.args[0]
        if fam is None:
            continue
        if not _linear_in_x(arg):
            return "other", f"{fam} atom with non-linear argument"
        fams.add(fam)
        subs[node] = sp.Dummy()

    if not fams:
        return "other", "no supported function family"
    if len(fams) > 1:
        return "other", "mixes families: " + "+".join(sorted(fams))
    residual = expr.xreplace(subs)
    if residual.has(x):
        return "other", "x appears outside the function atoms"
    if not _is_nonconstant(expr):
        return "other", "numerically constant"
    return fams.pop(), "single family, linear argument"


def classify(text: str):
    """Classify raw text. Returns (label, info_dict). Never raises."""
    try:
        s, expr = parse(text)
    except ParseError as ex:
        return "invalid", {"reason": str(ex)}
    label, reason = classify_expr(expr)
    return label, {"normalized": s, "sympy": str(expr), "reason": reason}


if __name__ == "__main__":
    for t in ["y = 2x² - 4x + 1", "2^x", "3*x + 5", "log(x)", "sin(3*x)",
              "x*sin(x)", "x**3", "\\frac{1}{2}x^{2}", "(x+3)*(x-5)",
              "__import__('os')", "exp(x)+x**2"]:
        print(f"{t!r:28} -> {classify(t)}")


Overwriting mathlabel.py


In [ ]:
%%writefile generate_dataset.py
"""
Step 1: Generate a labeled dataset of math expressions.

Each row: expression, label, template (which pattern produced it), fmt (surface style).
Surface formatting varies (spaces, ^ vs **, implicit multiplication) so the model
can't just memorize one style.
"""
import random
import re
import pandas as pd
from mathlabel import classify, normalize

random.seed(42)

def coef(lo=-9, hi=9, nonzero=True):
    while True:
        v = random.randint(lo, hi)
        if v != 0 or not nonzero:
            return v

def fcoef():
    """Sometimes a decimal or fraction-like coefficient."""
    r = random.random()
    if r < 0.7:
        return str(coef())
    if r < 0.9:
        return str(round(random.uniform(0.2, 9.5), 1))
    return f"{coef(1, 9)}/{coef(2, 9)}"

def sign_join(terms):
    """Join terms, turning '+ -3' into '- 3'."""
    out = terms[0]
    for t in terms[1:]:
        if t.startswith("-"):
            out += f" - {t[1:]}"
        else:
            out += f" + {t}"
    return out

def const_tail():
    return random.choice([None, str(coef())])

# ---------- templates: each returns an expression string in python-ish syntax ----------
def linear_templates():
    def t_slope_int():
        return sign_join([f"{coef()}*x", str(coef())])
    def t_slope_only():
        return f"{coef()}*x"
    def t_x_plus_c():
        return sign_join(["x", str(coef())])
    def t_neg_x():
        return sign_join(["-x", str(coef())])
    def t_fraction():
        return sign_join([f"x/{coef(2, 9)}", str(coef())])
    def t_factored():  # a*(x+b) is still linear
        return f"{coef()}*(x {random.choice(['+','-'])} {coef(1, 9)})"
    def t_decimal():
        return sign_join([f"{round(random.uniform(0.1, 9.9), 1)}*x", str(coef())])
    def t_constant_shift():  # 2*(x+1) - 3*(x-2)  -> still linear
        a = coef()
        b = random.choice([v for v in range(1, 6) if v != a])
        return f"{a}*(x + {coef(1,5)}) - {b}*(x - {coef(1,5)})"
    return [("lin_slope_int", t_slope_int), ("lin_slope_only", t_slope_only),
            ("lin_x_plus_c", t_x_plus_c), ("lin_neg_x", t_neg_x),
            ("lin_fraction", t_fraction), ("lin_factored", t_factored),
            ("lin_decimal", t_decimal), ("lin_collapse", t_constant_shift)]

def quadratic_templates():
    def t_full():
        return sign_join([f"{coef()}*x**2", f"{coef()}*x", str(coef())])
    def t_pure():
        return f"{coef()}*x**2"
    def t_no_linear():
        return sign_join([f"{coef()}*x**2", str(coef())])
    def t_no_const():
        return sign_join([f"{coef()}*x**2", f"{coef()}*x"])
    def t_vertex():
        return f"{coef()}*(x {random.choice(['+','-'])} {coef(1, 9)})**2 {random.choice(['+','-'])} {coef(1, 9)}"
    def t_factored():  # (x+a)*(x+b)
        return f"(x {random.choice(['+','-'])} {coef(1, 9)})*(x {random.choice(['+','-'])} {coef(1, 9)})"
    def t_bare():
        return random.choice(["x**2", "x*x", "-x**2"]) + random.choice(["", f" + {coef(1,9)}", f" - {coef(1,9)}"])
    def t_decimal():
        return sign_join([f"{round(random.uniform(0.1, 9.9), 1)}*x**2", f"{coef()}*x", str(coef())])
    return [("quad_full", t_full), ("quad_pure", t_pure), ("quad_no_linear", t_no_linear),
            ("quad_no_const", t_no_const), ("quad_vertex", t_vertex),
            ("quad_factored", t_factored), ("quad_bare", t_bare), ("quad_decimal", t_decimal)]

def exponential_templates():
    def t_exp():
        return random.choice([f"exp(x)", f"exp({coef()}*x)", f"{coef()}*exp(x)", f"{coef()}*exp({coef()}*x)"])
    def t_base():
        return f"{coef(2, 9)}**x"
    def t_coef_base():
        return f"{coef()}*{coef(2, 9)}**x"
    def t_shift():
        return sign_join([f"{coef(2, 9)}**x", str(coef())])
    def t_decay():
        return random.choice([f"exp(-x)", f"exp(-{coef(1,5)}*x)", f"0.{random.randint(2,9)}**x"])
    def t_exp_shift():
        return sign_join([f"{coef()}*exp({coef()}*x)", str(coef())])
    def t_growth():
        return f"{coef(1,9)}*(1.{random.randint(1,9)})**x"
    def t_e_pow():
        return random.choice(["e**x", f"e**({coef()}*x)", f"{coef()}*e**x"])
    return [("exp_exp", t_exp), ("exp_base", t_base), ("exp_coef_base", t_coef_base),
            ("exp_shift", t_shift), ("exp_decay", t_decay), ("exp_exp_shift", t_exp_shift),
            ("exp_growth", t_growth), ("exp_e_pow", t_e_pow)]

def logarithmic_templates():
    def t_ln():
        return random.choice(["log(x)", "ln(x)", f"{coef()}*log(x)", f"{coef()}*ln(x)"])
    def t_shift_in():
        return f"log(x {random.choice(['+','-'])} {coef(1, 9)})"
    def t_shift_out():
        return sign_join([f"{coef()}*log(x)", str(coef())])
    def t_base10():
        return random.choice(["log10(x)", f"{coef()}*log10(x)", f"log10(x) + {coef(1,9)}"])
    def t_base2():
        return random.choice(["log2(x)", f"{coef()}*log2(x)", f"log2(x) - {coef(1,9)}"])
    def t_scaled_arg():
        return f"log({coef(2, 9)}*x)"
    def t_full():
        return f"{coef()}*log(x {random.choice(['+','-'])} {coef(1,9)}) {random.choice(['+','-'])} {coef(1,9)}"
    def t_log_base():
        return f"log(x, {coef(2, 9)})"
    return [("log_ln", t_ln), ("log_shift_in", t_shift_in), ("log_shift_out", t_shift_out),
            ("log_base10", t_base10), ("log_base2", t_base2), ("log_scaled_arg", t_scaled_arg),
            ("log_full", t_full), ("log_base", t_log_base)]

def trig_templates():
    fn = lambda: random.choice(["sin", "cos", "tan"])
    def t_plain():
        return f"{fn()}(x)"
    def t_amp():
        return f"{coef()}*{fn()}(x)"
    def t_freq():
        return f"{fn()}({coef(1, 9)}*x)"
    def t_full():
        return f"{coef()}*{fn()}({coef(1,6)}*x {random.choice(['+','-'])} {coef(1,6)}) {random.choice(['+','-'])} {coef(1,6)}"
    def t_shift():
        return sign_join([f"{coef()}*{fn()}(x)", str(coef())])
    def t_pi():
        return random.choice([f"sin(pi*x)", f"cos(2*pi*x)", f"{coef()}*sin(x/{coef(2,6)})"])
    def t_sum():
        return f"sin(x) + cos(x)"
    def t_sq():
        return random.choice(["sin(x)**2", "cos(x)**2"])
    return [("trig_plain", t_plain), ("trig_amp", t_amp), ("trig_freq", t_freq),
            ("trig_full", t_full), ("trig_shift", t_shift), ("trig_pi", t_pi),
            ("trig_sum", t_sum), ("trig_sq", t_sq)]

def other_templates():
    """Things that are NOT one of the five supported families (or mix them)."""
    def t_cubic():
        return sign_join([f"{coef()}*x**3", f"{coef()}*x**2", f"{coef()}*x", str(coef())])
    def t_simple_pow():
        return random.choice([f"x**{random.randint(3, 6)}", f"{coef()}*x**{random.randint(3, 5)}"])
    def t_sqrt():
        return random.choice(["sqrt(x)", f"{coef()}*sqrt(x)", f"sqrt(x + {coef(1,9)})", "x**0.5"])
    def t_recip():
        return random.choice(["1/x", f"{coef()}/x", f"1/(x + {coef(1,9)})", f"{coef()}/(x - {coef(1,9)})"])
    def t_abs():
        return random.choice(["abs(x)", f"abs(x - {coef(1,9)})", f"{coef()}*abs(x) + {coef()}"])
    def t_x_trig():
        return random.choice(["x*sin(x)", "x*cos(x)", f"{coef()}*x*sin(x)", "x**2*cos(x)"])
    def t_x_exp():
        return random.choice(["x*exp(x)", f"x*{coef(2,5)}**x", "x**2*exp(x)", "x*exp(-x)"])
    def t_x_log():
        return random.choice(["x*log(x)", f"x**2*log(x)", f"log(x)/x"])
    def t_mixed():
        return random.choice(["exp(x) + x**2", "sin(x) + x", "log(x) + exp(x)",
                              "sin(x) + log(x)", f"x + {coef(2,5)}**x", "x**2 + sin(x)",
                              "cos(x) + exp(x)", "log(x) + x**2"])
    def t_inner_nonlinear():
        return random.choice(["sin(x**2)", "exp(x**2)", "log(x**2 + 1)", "cos(x**2)", "exp(sin(x))"])
    def t_constant():
        return random.choice([str(coef()), f"{coef()}.5", "pi", "e"])
    def t_poly_mix():
        return random.choice([f"x**3 - {coef(1,9)}*x", f"x**4 - {coef(1,9)}", f"(x + {coef(1,5)})**3"])
    return [("oth_cubic", t_cubic), ("oth_pow", t_simple_pow), ("oth_sqrt", t_sqrt),
            ("oth_recip", t_recip), ("oth_abs", t_abs), ("oth_x_trig", t_x_trig),
            ("oth_x_exp", t_x_exp), ("oth_x_log", t_x_log), ("oth_mixed", t_mixed),
            ("oth_inner", t_inner_nonlinear), ("oth_const", t_constant), ("oth_poly", t_poly_mix)]

CLASSES = {
    "linear": linear_templates(),
    "quadratic": quadratic_templates(),
    "exponential": exponential_templates(),
    "logarithmic": logarithmic_templates(),
    "trigonometric": trig_templates(),
    "other": other_templates(),
}

# ---------- surface-format noise ----------
def _to_unicode(expr):
    """x**2 -> x², 3*x -> 3x, only if exponents are 2/3 (else leave python style)."""
    out = re.sub(r"(\d)\*([a-z(])", r"\1\2", expr)
    if re.search(r"\*\*(?![23](?!\d))", out):
        return None
    return out.replace("**2", "²").replace("**3", "³")

def _to_latex(expr):
    out = expr.replace("ln(", "\\ln(")
    out = re.sub(r"\b(sin|cos|tan|log|exp|sqrt)\b", r"\\\1", out)
    out = re.sub(r"\\\\ln", r"\\ln", out)
    # exponents first, so "**" is never split into two \\cdot
    out = re.sub(r"\*\*(\([^()]*\)|-?[\w.]+)", lambda m: "^{" + m.group(1).strip("()") + "}", out)
    out = re.sub(r"(\d)\*([a-z(\\])", r"\1\2", out)
    return out.replace("*", " \\cdot ")

def apply_format(expr):
    """Return (formatted_expr, fmt_name). Same math, different typing habits."""
    r = random.random()
    if r < 0.38:
        return expr, "python"
    if r < 0.55:
        return expr.replace("**", "^"), "caret"
    if r < 0.67:
        return expr.replace(" ", ""), "nospace"
    if r < 0.79:
        out = re.sub(r"(\d)\*([a-z(])", r"\1\2", expr)
        return out.replace("**", "^"), "implicit"
    if r < 0.88:
        u = _to_unicode(expr)
        if u is not None:
            return u, "unicode"
    if r < 0.94:
        if "log(" not in expr or "," not in expr:  # keep log(x, b) out of latex
            return _to_latex(expr), "latex"
    prefix = random.choice(["y = ", "f(x) = ", "y="])
    return prefix + expr.replace("**", "^"), "prefixed"

def generate(n_per_class=1000, validate=True):
    """Generate rows. With validate=True every row is checked by the sympy referee;
    disagreements are dropped and returned separately for inspection."""
    rows, rejected, seen_canon = [], [], set()
    for label, templates in CLASSES.items():
        seen, kept, attempts = set(), 0, 0
        while kept < n_per_class and attempts < n_per_class * 30:
            attempts += 1
            name, fn = random.choice(templates)
            expr, fmt = apply_format(fn())
            if expr in seen:
                continue
            seen.add(expr)
            row = {"expression": expr, "label": label, "template": name,
                   "fmt": fmt, "source": "generated"}
            if validate:
                pred, info = classify(expr)
                if pred != label:
                    rejected.append({**row, "referee": pred, "reason": info.get("reason")})
                    continue
                canon = info["sympy"]
                if canon in seen_canon:      # same math, different typing -> drop
                    continue
                seen_canon.add(canon)
                row["canonical"] = canon
            rows.append(row)
            kept += 1
    df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
    return df, pd.DataFrame(rejected)

if __name__ == "__main__":
    df, rejected = generate(1000)
    df.to_csv("math_expressions.csv", index=False)
    rejected.to_csv("rejected_expressions.csv", index=False)
    print(f"Saved {len(df)} rows to math_expressions.csv")
    print(f"Referee rejected {len(rejected)} candidate rows (see rejected_expressions.csv)\n")
    print("Class balance:\n", df["label"].value_counts(), "\n")
    print("Formats:\n", df["fmt"].value_counts(normalize=True).round(2), "\n")
    print("Duplicate expressions:", df["expression"].duplicated().sum())
    # normalized duplicates: different typing, same math text
    print("Duplicate canonical forms:", df["canonical"].duplicated().sum(), "\n")
    print("Sample rows:")
    print(df.groupby("label").head(2).sort_values("label").to_string(index=False))


Overwriting generate_dataset.py


In [ ]:
%%writefile test_mathlabel.py
"""Run: python test_mathlabel.py   (plain asserts, no pytest needed)"""
from mathlabel import classify, normalize

CASES = [
    # (input, expected)
    ("3*x + 5", "linear"), ("-x", "linear"), ("x/3 + 1", "linear"),
    ("2*(x+1) - 3*(x-2)", "linear"),            # collapses to -x + 8
    ("2*(x+1) - 2*(x-2)", "other"),             # collapses to constant 6
    ("2x² - 4x + 1", "quadratic"), ("(x+3)*(x-5)", "quadratic"), ("x*x", "quadratic"),
    ("y = 3x^2 + 2x + 1", "quadratic"), ("\\frac{1}{2}x^{2}", "quadratic"),
    ("x**2 - (x-1)**2", "linear"),              # x² cancels -> 2x - 1
    ("2^x", "exponential"), ("exp(-3*x) + 4", "exponential"), ("e**(2x)", "exponential"),
    ("0.5**x", "exponential"), ("(-2)**x", "other"), ("1**x", "other"),
    ("log(x)", "logarithmic"), ("ln(x+1)", "logarithmic"), ("log10(x) + 2", "logarithmic"),
    ("log(x, 3)", "logarithmic"), ("\\ln(x)", "logarithmic"),
    ("sin(3x)", "trigonometric"), ("2cos(x) - 1", "trigonometric"),
    ("sin(x)**2", "trigonometric"), ("\\sin(2 x)", "trigonometric"), ("cos(2*pi*x)", "trigonometric"),
    ("sin(x)**2 + cos(x)**2", "other"),         # identically 1
    ("x**3", "other"), ("sqrt(x)", "other"), ("1/x", "other"), ("abs(x)", "other"),
    ("x*sin(x)", "other"), ("exp(x) + x**2", "other"), ("sin(x) + log(x)", "other"),
    ("log(x**2)", "other"),                     # even function, not a standard log curve
    ("sin(x**2)", "other"), ("exp(sin(x))", "other"), ("7", "other"), ("pi", "other"),
    # hostile / garbage input must never execute or crash
    ("__import__('os').system('ls')", "invalid"), ("x; import os", "invalid"),
    ("open('f')", "invalid"), ("a*x + b", "invalid"), ("", "invalid"),
    ("3*x +", "invalid"), ("x" * 500, "invalid"),
]

fails = 0
for text, want in CASES:
    got, info = classify(text)
    if got != want:
        fails += 1
        print(f"FAIL {text!r}: expected {want}, got {got} ({info})")
print(f"{len(CASES) - fails}/{len(CASES)} passed")
assert normalize("y = 2x²") == "2x**2"
raise SystemExit(1 if fails else 0)


Overwriting test_mathlabel.py


## Sanity check: referee tests
Expect `47/47 passed`.

In [ ]:
!python test_mathlabel.py

47/47 passed


## Generate the dataset
Every row is checked by the referee; disagreements are dropped into `rejected_expressions.csv`.

In [ ]:
from generate_dataset import generate
df, rejected = generate(1000)
df.to_csv('math_expressions.csv', index=False)
print(len(df), 'rows |', len(rejected), 'rejected by referee')
df.sample(10, random_state=0)

6000 rows | 0 rejected by referee


,expression,label,template,fmt,source,canonical
4320,5(x + 9)^2 + 4,quadratic,quad_vertex,implicit,generated,5*(x + 9)**2 + 4
2006,5**x - 4,exponential,exp_shift,python,generated,5**x - 4
5689,-9*cos(x) + 4,trigonometric,trig_shift,python,generated,4 - 9*cos(x)
472,y = 1*log(x) + 5,logarithmic,log_shift_out,prefixed,generated,log(x) + 5
1370,7\sin(4x + 4) - 5,trigonometric,trig_full,latex,generated,7*sin(4*x + 4) - 5
1457,7^{x} + 6,exponential,exp_shift,latex,generated,7**x + 6
5783,-6*cos(2*x + 5) - 4,trigonometric,trig_full,caret,generated,-6*cos(2*x + 5) - 4
3484,-7*3^x,exponential,exp_coef_base,caret,generated,-7*3**x
1145,-5*cos(1*x - 3) - 6,trigonometric,trig_full,python,generated,-5*cos(x - 3) - 6
130,8*cos(2*x - 3) + 3,trigonometric,trig_full,python,generated,8*cos(2*x - 3) + 3


## Explore (step 2)

In [ ]:
import pandas as pd
print(df['label'].value_counts(), '\n')
print('Exact duplicates:    ', df['expression'].duplicated().sum())
print('Canonical duplicates:', df['canonical'].duplicated().sum())
pd.crosstab(df['label'], df['fmt'])

label
quadratic        1000
logarithmic      1000
linear           1000
exponential      1000
other            1000
trigonometric    1000
Name: count, dtype: int64 

Exact duplicates:     0
Canonical duplicates: 0


fmt,caret,implicit,latex,nospace,prefixed,python,unicode
label,,,,,,,
exponential,168,128,95,126,57,366,60
linear,148,120,58,133,61,395,85
logarithmic,169,133,52,123,52,370,101
other,181,128,56,120,65,360,90
quadratic,159,118,54,133,62,404,70
trigonometric,166,131,57,132,58,382,74


In [ ]:
# Try the classifier on anything you like
from mathlabel import classify
for t in ['y = 2x² - 4x + 1', '2^x', 'sin(3x)', 'x*sin(x)', '\\frac{1}{2}x^{2}', 'a*x + b']:
    label, info = classify(t)
    print(f'{t!r:24} -> {label:14} ({info["reason"]})')

'y = 2x² - 4x + 1'       -> quadratic      (polynomial degree 2)
'2^x'                    -> exponential    (single family, linear argument)
'sin(3x)'                -> trigonometric  (single family, linear argument)
'x*sin(x)'               -> other          (x appears outside the function atoms)
'\\frac{1}{2}x^{2}'      -> quadratic      (polynomial degree 2)
'a*x + b'                -> invalid        (name not allowed: a)
